# 05 — LSTM (EXTRA, dia 18 se houver folga)

Experimento comparativo: uma LSTM lê a **sequência das últimas 72 h de chuva** da célula do
ponto e junta com as features fixas/calendário. Perder para o LightGBM **também é resultado
válido** (e é o esperado com poucos positivos).

Instalação: `pip install -r requirements-extra.txt` (PyTorch, CPU basta).

Cuidados para a comparação ser justa:
- mesma divisão temporal, mesmas linhas (inclusive a subamostragem do treino com peso);
- mesma calibração (`src.calibracao.Calibrador`) e mesmas métricas (`src.evaluate.avaliar`).

In [ ]:
# Preparação: permite importar src/ a partir da pasta notebooks/
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config

pd.set_option("display.max_columns", 50)
print("Raiz do projeto:", RAIZ)

In [ ]:
import json

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from src.dataset import carregar
from src.features import COLUNAS_TEMPO

VERSAO = "v1"
JANELA = 72
dataset, resumo = carregar(VERSAO)
chuva = pd.read_parquet(config.ARQ_CHUVA_HORARIA)          # celula_id × hora
pontos = pd.read_csv(config.DIR_PROCESSED / f"dataset_{VERSAO}_pontos.csv")
celula_do_ponto = dict(zip(pontos.ponto_id, pontos.celula_id))
torch.manual_seed(config.SEMENTE)

## Sequências
Para cada linha `(ponto, t)`, os 72 valores de chuva que terminam em `t` (mesma convenção das features).

In [ ]:
def matriz_sequencias(df: pd.DataFrame) -> np.ndarray:
    """(n_linhas, 72) com log1p da chuva horária até t, por célula."""
    saida = np.zeros((len(df), JANELA), dtype="float32")
    df = df.reset_index(drop=True)
    celulas = df["ponto_id"].astype(str).map(celula_do_ponto)
    for celula, idx in df.groupby(celulas).groups.items():
        s = chuva[chuva.celula_id == celula].set_index("data_hora_utc")["chuva_mm"].sort_index()
        s = s.reindex(pd.date_range(s.index.min(), s.index.max(), freq="h"))
        valores = np.log1p(s.fillna(0).to_numpy(dtype="float32"))
        pos = s.index.get_indexer(pd.DatetimeIndex(df.loc[idx, "data_hora_utc"]))
        janelas = pos[:, None] + np.arange(-JANELA + 1, 1)[None, :]
        saida[np.asarray(idx)] = valores[janelas]
    return saida

ESTATICAS = COLUNAS_TEMPO + ["elevacao", "dist_corrego_m", "freq_hist_alagamento"]

def matriz_estatica(df, media=None, desvio=None):
    X = df[ESTATICAS].astype("float32").copy()
    X = pd.concat([X, pd.get_dummies(df["zona"].astype("category").cat.set_categories(config.ZONAS), dtype="float32")], axis=1)
    X = X.fillna(0 if media is None else media)
    media = X.mean() if media is None else media
    desvio = X.std().replace(0, 1) if desvio is None else desvio
    return ((X - media) / desvio).to_numpy(dtype="float32"), media, desvio

partes = {n: dataset[dataset.split == n].reset_index(drop=True) for n in ("treino", "validacao", "teste")}
seq = {n: matriz_sequencias(d) for n, d in partes.items()}
est_treino, MEDIA, DESVIO = matriz_estatica(partes["treino"])
est = {"treino": est_treino, **{n: matriz_estatica(partes[n], MEDIA, DESVIO)[0] for n in ("validacao", "teste")}}
{n: s.shape for n, s in seq.items()}

In [ ]:
class Linhas(Dataset):
    def __init__(self, nome):
        self.seq = torch.from_numpy(seq[nome]).unsqueeze(-1)
        self.est = torch.from_numpy(est[nome])
        self.y = torch.from_numpy(partes[nome]["alagou_3h"].to_numpy("float32"))
        self.w = torch.from_numpy(partes[nome]["peso_amostra"].to_numpy("float32"))
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.seq[i], self.est[i], self.y[i], self.w[i]


class LSTMAlagamento(nn.Module):
    def __init__(self, n_estaticas, oculto=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=oculto, batch_first=True)
        self.cabeca = nn.Sequential(nn.Linear(oculto + n_estaticas, 32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, 1))
    def forward(self, seq, est):
        _, (h, _) = self.lstm(seq)
        return self.cabeca(torch.cat([h[-1], est], dim=1)).squeeze(-1)

In [ ]:
from sklearn.metrics import average_precision_score

modelo = LSTMAlagamento(est["treino"].shape[1])
otimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)
n_pos = partes["treino"].alagou_3h.sum()
pos_weight = torch.tensor(min(50.0, (len(partes["treino"]) - n_pos) / max(n_pos, 1)))
perda = nn.BCEWithLogitsLoss(reduction="none", pos_weight=pos_weight)
carregador = DataLoader(Linhas("treino"), batch_size=2048, shuffle=True)
val = Linhas("validacao")

def pontuar(ds):
    modelo.eval()
    with torch.no_grad():
        saida = [torch.sigmoid(modelo(ds.seq[i:i + 20000], ds.est[i:i + 20000])) for i in range(0, len(ds), 20000)]
    return torch.cat(saida).numpy()

melhor, paciencia = 0.0, 0
for epoca in range(30):
    modelo.train()
    for s, e, y, w in carregador:
        otimizador.zero_grad()
        (perda(modelo(s, e), y) * w).mean().backward()
        otimizador.step()
    ap = average_precision_score(val.y.numpy(), pontuar(val))
    print(f"época {epoca}: PR-AUC val = {ap:.4f}")
    if ap > melhor:
        melhor, paciencia, estado = ap, 0, {k: v.clone() for k, v in modelo.state_dict().items()}
    else:
        paciencia += 1
        if paciencia >= 4:
            break
modelo.load_state_dict(estado)

In [ ]:
from src.calibracao import Calibrador
from src.evaluate import avaliar, tabela_comparacao
from src.niveis import niveis_por_probabilidade

cal = Calibrador("auto").ajustar(pontuar(val), val.y.numpy())
teste = Linhas("teste")
p_teste = cal.transformar(pontuar(teste))
lim = config.LIMIARES_NIVEIS_PADRAO
res = {"lstm": avaliar(partes["teste"], p_teste, niveis_por_probabilidade(p_teste, lim))}
metricas = json.loads((config.DIR_RELATORIOS / f"metricas_{VERSAO}.json").read_text()) if (config.DIR_RELATORIOS / f"metricas_{VERSAO}.json").exists() else {"teste": {}}
pd.concat([tabela_comparacao(metricas["teste"]), tabela_comparacao(res)])

Registre o resultado (mesmo que a LSTM perca) no TCC: arquitetura, épocas, PR-AUC, Brier e tempo de treino.